Data obtained here: https://zenodo.org/records/14767363
egrid details here: https://www.epa.gov/system/files/documents/2025-01/egrid2023_technical_guide.pdf
ejscreen in action here: https://pedp-ejscreen.azurewebsites.net/
ejscreen documentation here: https://www.epa.gov/system/files/documents/2024-07/ejscreen-tech-doc-version-2-3.pdf
other resource i couldnt figure out: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/RLR5AX
ejscreen tool archive: https://screening-tools.com/epa-ejscreen


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import seaborn as sns
import pymc as pm
import bambi as bmb
import arviz as az
import statsmodels.api as sm
from sklearn.preprocessing import OneHotEncoder


In [10]:
import pandas as pd


use_cols = [
    'ID', 
    'PEOPCOLOR', 'ACSTOTPOP',
    'LOWINCOME', 'ACSIPOVBAS',
    'UNEMPLOYED', 'ACSUNEMPBAS',
    'LINGISO', 'ACSTOTHH',
    'LESSHS', 'ACSEDUCBAS',
    'UNDER5', 'OVER64',
    'P_LIFEEXPPCT', 'ST_ABBREV', 'CNTY_NAME'
]

# 2. Initialize an empty list to store our mini-results
chunk_results = []

#Processing files in chunks because it was crashing kernel
chunk_size = 5000
file_path = 'data/EJSCREEN_2023_BG_with_AS_CNMI_GU_VI.csv'

# need latin1 because the file has special characters
with pd.read_csv(file_path, chunksize=chunk_size, usecols=use_cols, encoding='latin1', low_memory=False) as reader:
    for chunk in reader:
        # Create County FIPS on the fly
        chunk['County FIPS'] = chunk['ID'].astype(str).str.zfill(12).str[:5]
        
        # Aggregate JUST this chunk by County FIPS
        # (It's okay if a county is split across chunks; we will sum them again at the end)
        agg_chunk = chunk.groupby('County FIPS')[use_cols[1:]].sum()
        
        # Store this small aggregated piece
        chunk_results.append(agg_chunk)



In [23]:
# 1. Concatenate all chunks
concatenated = pd.concat(chunk_results)

# 2. Define how to handle each column
# 'first' means: just grab the name/state from the first row you see for this FIPS
# 'sum' means: add up all the numbers
agg_rules = {
    'ST_ABBREV': 'first',
    'CNTY_NAME': 'first',
    'P_LIFEEXPPCT': 'first',
    # Apply 'sum' to all your numeric columns automatically:
    'PEOPCOLOR': 'sum',
    'ACSTOTPOP': 'sum',
    'LOWINCOME': 'sum',
    'ACSIPOVBAS': 'sum',
    'UNEMPLOYED': 'sum',
    'ACSUNEMPBAS': 'sum',
    'LINGISO': 'sum', 'ACSTOTHH': 'sum',
    'LESSHS': 'sum', 'ACSEDUCBAS': 'sum',
    'UNDER5': 'sum', 'OVER64': 'sum',
 
}


# OR, a cleaner dynamic way if you don't want to type every column name:
agg_rules = {col: 'sum' for col in use_cols[1:]}
agg_rules['ST_ABBREV'] = 'first'
agg_rules['CNTY_NAME'] = 'first'
agg_rules['P_LIFEEXPPCT'] = 'first'

# 3. Perform the final GroupBy
final_df = concatenated.groupby(level=0).agg(agg_rules).reset_index()

In [24]:


# 4. Concatenate all the small pieces and sum them one last time
# This combines the data from counties that were split across different chunks
#final_df = pd.concat(chunk_results).groupby(level=0).sum().reset_index()

# 5. Now calculate your percentages on the final, small dataframe
final_df['Total Population'] = final_df['ACSTOTPOP']
final_df['People of Color (%)'] = (final_df['PEOPCOLOR'] / final_df['ACSTOTPOP']) * 100
final_df['Low Income (%)'] = (final_df['LOWINCOME'] / final_df['ACSIPOVBAS']) * 100
final_df['Unemployment Rate (%)'] = (final_df['UNEMPLOYED'] / final_df['ACSUNEMPBAS']) * 100
final_df['Limited English Speaking (%)'] = (final_df['LINGISO'] / final_df['ACSTOTHH']) * 100
final_df['Less Than High School Education (%)'] = (final_df['LESSHS'] / final_df['ACSEDUCBAS']) * 100
final_df['Under Age 5 (%)'] = (final_df['UNDER5'] / final_df['ACSTOTPOP']) * 100
final_df['Over Age 64 (%)'] = (final_df['OVER64'] / final_df['ACSTOTPOP']) * 100
final_df['Plant state abbreviation'] = final_df['ST_ABBREV']
final_df['Plant county name'] = final_df['CNTY_NAME']


#missing : Limited Life Expectancy (%)  
final_df

,County FIPS,ST_ABBREV,CNTY_NAME,P_LIFEEXPPCT,PEOPCOLOR,ACSTOTPOP,LOWINCOME,ACSIPOVBAS,UNEMPLOYED,ACSUNEMPBAS,...,Total Population,People of Color (%),Low Income (%),Unemployment Rate (%),Limited English Speaking (%),Less Than High School Education (%),Under Age 5 (%),Over Age 64 (%),Plant state abbreviation,Plant county name
0,00000,MPMPMPMPMPMPMPASASMPMPASMPASGUGUASASASASASASAS...,Saipan MunicipalitySaipan MunicipalityTinian M...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MPMPMPMPMPMPMPASASMPMPASMPASGUGUASASASASASASAS...,Saipan MunicipalitySaipan MunicipalityTinian M...
1,00780,VIVIVIVIVIVIVIVIVIVIVIVIVIVIVIVIVIVIVIVIVIVIVI...,St. Croix IslandSt. Croix IslandSt. Croix Isla...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VIVIVIVIVIVIVIVIVIVIVIVIVIVIVIVIVIVIVIVIVIVIVI...,St. Croix IslandSt. Croix IslandSt. Croix Isla...
2,01001,ALALALALALALALALALALALALALALALALALALALALALALAL...,Autauga CountyAutauga CountyAutauga CountyAuta...,2883.0,15668.0,58239.0,17782.0,57790.0,752.0,26623.0,...,58239.0,26.902934,30.770029,2.824625,0.146413,10.415510,5.697213,15.135905,ALALALALALALALALALALALALALALALALALALALALALALAL...,Autauga CountyAutauga CountyAutauga CountyAuta...
3,01003,ALALALALALALALALALALALALALALALALALALALALALALAL...,Baldwin CountyBaldwin CountyBaldwin CountyBald...,6203.0,39583.0,227131.0,57840.0,223772.0,3994.0,108361.0,...,227131.0,17.427388,25.847738,3.685828,0.837252,8.985844,5.298704,20.607051,ALALALALALALALALALALALALALALALALALALALALALALAL...,Baldwin CountyBaldwin CountyBaldwin CountyBald...
4,01005,ALALALALALALALALALALALALALALALALALALALALALAL,Barbour CountyBarbour CountyBarbour CountyBarb...,1438.0,13991.0,25259.0,11195.0,22250.0,808.0,9369.0,...,25259.0,55.390158,50.314607,8.624186,1.287412,24.328980,5.225860,19.007087,ALALALALALALALALALALALALALALALALALALALALALAL,Barbour CountyBarbour CountyBarbour CountyBarb...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3218,72145,PRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPR...,Vega Baja MunicipioVega Baja MunicipioVega Baj...,0.0,53453.0,54544.0,40080.0,54242.0,3482.0,19789.0,...,54544.0,97.999780,73.891081,17.595634,67.427648,23.884740,4.018774,21.016060,PRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPR...,Vega Baja MunicipioVega Baja MunicipioVega Baj...
3219,72147,PRPRPRPRPRPRPR,Vieques MunicipioVieques MunicipioVieques Muni...,0.0,7808.0,8317.0,7186.0,8317.0,358.0,2355.0,...,8317.0,93.880005,86.401347,15.201699,75.063184,27.018425,4.821450,22.892870,PRPRPRPRPRPRPR,Vieques MunicipioVieques MunicipioVieques Muni...
3220,72149,PRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPR,Villalba MunicipioVillalba MunicipioVillalba M...,0.0,22289.0,22341.0,17790.0,22207.0,1464.0,7856.0,...,22341.0,99.767244,80.109875,18.635438,73.884699,21.445597,4.485028,18.745804,PRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPR,Villalba MunicipioVillalba MunicipioVillalba M...
3221,72151,PRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPR...,Yabucoa MunicipioYabucoa MunicipioYabucoa Muni...,0.0,31020.0,31047.0,24486.0,31042.0,1506.0,9897.0,...,31047.0,99.913035,78.880227,15.216732,72.616548,26.870868,3.517248,21.905498,PRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPRPR...,Yabucoa MunicipioYabucoa MunicipioYabucoa Muni...


In [29]:
import pandas as pd

use_cols = [
    'ID', 
    'PEOPCOLOR', 'ACSTOTPOP',
    'LOWINCOME', 'ACSIPOVBAS',
    'UNEMPLOYED', 'ACSUNEMPBAS',
    'LINGISO', 'ACSTOTHH',
    'LESSHS', 'ACSEDUCBAS',
    'UNDER5', 'OVER64',
    'P_LIFEEXPPCT', 'ST_ABBREV', 'CNTY_NAME'
]

chunk_results = []
chunk_size = 5000
file_path = 'data/EJSCREEN_2023_BG_with_AS_CNMI_GU_VI.csv'

# Define aggregation rules ONCE so we can use them in both the loop and the final step
# Note: P_LIFEEXPPCT is a Percentile, not a raw count. 
# Averaging percentiles is generally not recommended, but 'mean' is safer than 'sum' or 'first' if you must use it.
agg_rules = {
    'ST_ABBREV': 'first',
    'CNTY_NAME': 'first',
    'P_LIFEEXPPCT': 'mean', #get an average percentile
    'PEOPCOLOR': 'sum',
    'ACSTOTPOP': 'sum',
    'LOWINCOME': 'sum',
    'ACSIPOVBAS': 'sum',
    'UNEMPLOYED': 'sum',
    'ACSUNEMPBAS': 'sum',
    'LINGISO': 'sum', 
    'ACSTOTHH': 'sum',
    'LESSHS': 'sum', 
    'ACSEDUCBAS': 'sum',
    'UNDER5': 'sum', 
    'OVER64': 'sum',
}

with pd.read_csv(file_path, chunksize=chunk_size, usecols=use_cols, encoding='latin1', low_memory=False) as reader:
    for chunk in reader:
        # Create County FIPS on the fly
        chunk['County FIPS'] = chunk['ID'].astype(str).str.zfill(12).str[:5]
        
        # --- FIX IS HERE ---
        # Instead of .sum(), we use .agg(agg_rules) inside the loop too.
        # This prevents "AL" + "AL" = "ALAL"
        agg_chunk = chunk.groupby('County FIPS').agg(agg_rules)
        
        chunk_results.append(agg_chunk)

# 1. Concatenate all chunks
concatenated = pd.concat(chunk_results)

# 2. Perform the final GroupBy using the SAME rules
# We group by level=0 because 'County FIPS' is currently the index
final_df = concatenated.groupby(level=0).agg(agg_rules).reset_index()

# 3. Calculate Percentages
final_df['Total Population'] = final_df['ACSTOTPOP']
final_df['People of Color (%)'] = (final_df['PEOPCOLOR'] / final_df['ACSTOTPOP']) * 100
final_df['Low Income (%)'] = (final_df['LOWINCOME'] / final_df['ACSIPOVBAS']) * 100
final_df['Unemployment Rate (%)'] = (final_df['UNEMPLOYED'] / final_df['ACSUNEMPBAS']) * 100
final_df['Limited English Speaking (%)'] = (final_df['LINGISO'] / final_df['ACSTOTHH']) * 100
final_df['Less Than High School Education (%)'] = (final_df['LESSHS'] / final_df['ACSEDUCBAS']) * 100
final_df['Under Age 5 (%)'] = (final_df['UNDER5'] / final_df['ACSTOTPOP']) * 100
final_df['Over Age 64 (%)'] = (final_df['OVER64'] / final_df['ACSTOTPOP']) * 100
final_df['Plant state abbreviation'] = final_df['ST_ABBREV']
final_df['Plant county name'] = final_df['CNTY_NAME']

print(final_df.head())

  County FIPS ST_ABBREV            CNTY_NAME  P_LIFEEXPPCT  PEOPCOLOR  \
0       00000        MP  Saipan Municipality           NaN        0.0   
1       00780        VI     St. Croix Island           NaN        0.0   
2       01001        AL       Autauga County     72.075000    15668.0   
3       01003        AL       Baldwin County     56.908257    39583.0   
4       01005        AL       Barbour County     84.588235    13991.0   

   ACSTOTPOP  LOWINCOME  ACSIPOVBAS  UNEMPLOYED  ACSUNEMPBAS  ...  \
0        0.0        0.0         0.0         0.0          0.0  ...   
1        0.0        0.0         0.0         0.0          0.0  ...   
2    58239.0    17782.0     57790.0       752.0      26623.0  ...   
3   227131.0    57840.0    223772.0      3994.0     108361.0  ...   
4    25259.0    11195.0     22250.0       808.0       9369.0  ...   

   Total Population  People of Color (%)  Low Income (%)  \
0               0.0                  NaN             NaN   
1               0.0       

In [30]:
final_df

,County FIPS,ST_ABBREV,CNTY_NAME,P_LIFEEXPPCT,PEOPCOLOR,ACSTOTPOP,LOWINCOME,ACSIPOVBAS,UNEMPLOYED,ACSUNEMPBAS,...,Total Population,People of Color (%),Low Income (%),Unemployment Rate (%),Limited English Speaking (%),Less Than High School Education (%),Under Age 5 (%),Over Age 64 (%),Plant state abbreviation,Plant county name
0,00000,MP,Saipan Municipality,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MP,Saipan Municipality
1,00780,VI,St. Croix Island,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,VI,St. Croix Island
2,01001,AL,Autauga County,72.075000,15668.0,58239.0,17782.0,57790.0,752.0,26623.0,...,58239.0,26.902934,30.770029,2.824625,0.146413,10.415510,5.697213,15.135905,AL,Autauga County
3,01003,AL,Baldwin County,56.908257,39583.0,227131.0,57840.0,223772.0,3994.0,108361.0,...,227131.0,17.427388,25.847738,3.685828,0.837252,8.985844,5.298704,20.607051,AL,Baldwin County
4,01005,AL,Barbour County,84.588235,13991.0,25259.0,11195.0,22250.0,808.0,9369.0,...,25259.0,55.390158,50.314607,8.624186,1.287412,24.328980,5.225860,19.007087,AL,Barbour County
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3218,72145,PR,Vega Baja Municipio,NaN,53453.0,54544.0,40080.0,54242.0,3482.0,19789.0,...,54544.0,97.999780,73.891081,17.595634,67.427648,23.884740,4.018774,21.016060,PR,Vega Baja Municipio
3219,72147,PR,Vieques Municipio,NaN,7808.0,8317.0,7186.0,8317.0,358.0,2355.0,...,8317.0,93.880005,86.401347,15.201699,75.063184,27.018425,4.821450,22.892870,PR,Vieques Municipio
3220,72149,PR,Villalba Municipio,NaN,22289.0,22341.0,17790.0,22207.0,1464.0,7856.0,...,22341.0,99.767244,80.109875,18.635438,73.884699,21.445597,4.485028,18.745804,PR,Villalba Municipio
3221,72151,PR,Yabucoa Municipio,NaN,31020.0,31047.0,24486.0,31042.0,1506.0,9897.0,...,31047.0,99.913035,78.880227,15.216732,72.616548,26.870868,3.517248,21.905498,PR,Yabucoa Municipio


In [ ]:
final_df.to_csv('data/EJScreen_DEMO23.csv')